# M5 data quality and demand profile

This notebook documents the checks performed before modeling: data scale, zero demand, price coverage, store/category balance, and recent demand movement. It intentionally uses the production loader rather than a second notebook-only data pipeline.

In [ ]:
from pathlib import Path
from retail_planner.data.m5 import load_m5
from retail_planner.evaluation.diagnostics import profile_dataset

DATA_DIR = Path('../data/raw/m5')
SAMPLE_SIZE = 1_000
dataset = load_m5(DATA_DIR, limit_series=SAMPLE_SIZE)
profile = profile_dataset(dataset)
profile

In [ ]:
# Demand intermittency by category. Zero sales can mean low demand or unavailable stock.
category_rows = []
for category in sorted({item.cat_id for item in dataset.series}):
    rows = [item for item in dataset.series if item.cat_id == category]
    values = [value for item in rows for value in item.sales]
    category_rows.append({
        'category': category,
        'series': len(rows),
        'zero_share': sum(value == 0 for value in values) / len(values),
        'mean_units': sum(values) / len(values),
    })
category_rows

In [ ]:
# Recent level shifts are descriptive only; model features remain lagged at each origin.
shifts = []
for item in dataset.series:
    previous = sum(item.sales[-56:-28])
    recent = sum(item.sales[-28:])
    shifts.append((recent / previous - 1) if previous else None)
valid_shifts = sorted(value for value in shifts if value is not None)
{'median_shift': valid_shifts[len(valid_shifts)//2],
 'large_shift_share': sum(abs(value) > 0.25 for value in valid_shifts) / len(valid_shifts)}

## Findings to carry into modeling

- Demand is intermittent, so seasonal and Croston baselines are necessary checks.
- Store/category diagnostics must accompany global metrics.
- Prices and known calendar fields are useful only when available before the forecast target.
- A sampled study is not an official full-M5 submission; all published metrics retain that label.